# IAGOS Flight Search & Download for IRISCC (v2)

This notebook provides tools to search and download IAGOS flight data focused on **CO (Carbon Monoxide) from fire sources** using profile data (vertical measurements during ascent/descent).

If you are not familiar with Python, Notebooks or conda, please, feel free to read the QUICKSTART.md and README.md.
Then, for more details, you can have a look in flight_search_download_demo_v2.ipynb

## IRISCC Focus

This notebook is specifically designed for IRISCC with:
- **Fixed parameters**: `originCO = "fire"` and `type = "profiles"`
- **Variable parameters**: `start`, `end`, `bbox`, `level`, `region`

## Prerequisites

### Create a IAGOS account

1) Registrate following the link below: https://iagos.aeris-data.fr/registration/
2) Once it has been validated by the IAGOS team, you will be informed by receiving an email
3) On the authentication page https://iagos.aeris-data.fr/profile/, click on "forgot password" and create your password

### Generating Authentication Credentials

IAGOS services require authentication via AERIS SSO. You must generate a configuration file containing your credentials:

Open a terminal (click + near the current tab, in other, click "Terminal") and run the credential generation script by copying :

```bash 
cd /workspace/VREFolders/Wildfires_in-Europe/Iagos_vre
conda env list
conda activate iagos-notebooks
PYTHONPATH=src python src/fr/aeris/auth/generate_credentials.py
```

This script will prompt you for your AERIS credentials and create a secure configuration file.
Then, copy the lines provided in the authentication.py file line 25-26 as suggested in the terminal.



## Server Configuration

- **Production URL**: `https://services.iagos-data.fr/prod/v2.0/downloads`
- **Authentication**: Required via AERIS SSO

## API Services

### `/search` Service
Search for flights without downloading. Returns flight count and total data size.

### `/download-flights` Service
Download data files as a ZIP archive.
- **Maximum limit**: 1 GB per request
- **Validation**: Always search first to check data size

In [ ]:
#Install the libraries needed in the notebook. If it has already been done, you will receive messages saying "Requirement already satisfied"
#!pip install python-keycloak
#!pip install matplotlib
#!pip install h5netcdf
#!pip install h5py

# Add project source code to Python path
import sys
sys.path.insert(0, '../src')

# Import necessary libraries for this notebook
import json
import zipfile

# Import IAGOS authentication functions
from fr.aeris.auth import get_token_password_grant, AERIS_CONFIG, build_auth_header

# Import visualization libraries
import io
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("All imports successful")

from fr.aeris.iriscc import search_flights, download_flights, rank_flights_by_co

print("IRISCC utility functions imported successfully")

# API Configuration

BASE_URL = "https://services.iagos-data.fr/prod/v2.0/downloads"
print("Using REMOTE server: https://services.iagos-data.fr")
print(f"Base URL: {BASE_URL}")

In [ ]:
# Authentication with AERIS SSO (Single Sign-On)
#
# AERIS SSO is the authentication system used to access IAGOS data
# This step is required for downloading data or search L1 data

print("Authenticating with AERIS SSO...")

# get_token_password_grant() reads your credentials and requests an access token from AERIS
# The token is a temporary key that proves your identity
token_data = get_token_password_grant(AERIS_CONFIG)

# Extract the access token from the response
# This token will be included in HTTP requests to prove you are authorized
access_token = token_data['access_token']

# Build the authorization header in the format required by the API
# 'token' specifies the authentication method
auth_headers = build_auth_header('token', access_token)

print("Authentication successful!")
print(f"Token type: {token_data.get('token_type', 'Bearer')}")
print(f"Expires in: {token_data.get('expires_in', 'N/A')} seconds")

## API Reference for IRISCC

### Parameter Format

| Parameter | Type   | Required | Description                               | Example                         |
|-----------|--------|----------|-------------------------------------------|---------------------------------|
| start     | string | Yes      | Start date in ISO 8601 format             | `"2010-01-01T00:00:00"`         |
| end       | string | Yes      | End date in ISO 8601 format               | `"2020-12-31T23:59:59"`         |
| bbox      | string | Yes      | Bounding box: minLon,minLat,maxLon,maxLat | `"-180,-90,180,90"`             |
| level     | string | Yes      | Data level                                | `"L1"`, `"L2"` or `"L4"`        |
| region    | list   | No       | Region filter for CO source               | `["BONA", "EURO"]`              |
| l4Type    | list   | No       | L4 file types (L4 only)                   | `["ECMWF", "CO_CONTRIBUTIONS"]` |

**Fixed parameters for IRISCC:**
- `type = "profiles"` (vertical measurements during ascent/descent)
- `originCO = "fire"` (CO from biomass burning)

### L4 File Types

| Type             | Description                                                    |
|------------------|----------------------------------------------------------------|
| ECMWF            | Meteorological fields from ECMWF (temperature, wind, humidity) |
| CO_CONTRIBUTIONS | CO source attribution by region                                |
| PV               | Potential Vorticity                                            |

### Available Regions (GFED)

| Code | Region                            |
|------|-----------------------------------|
| AUST | Australia                         |
| BOAS | Boreal Asia                       |
| BONA | Boreal North America              |
| CEAM | Central America                   |
| CEAS | Central Asia                      |
| EQAS | Equatorial Asia                   |
| EURO | Europe                            |
| MIDE | Middle East                       |
| NHAF | Northern Hemisphere Africa        |
| NHSA | Northern Hemisphere South America |
| SEAS | Southeast Asia                    |
| SHAF | Southern Hemisphere Africa        |
| SHSA | Southern Hemisphere South America |
| TENA | Temperate North America           |

#### GFED Regions Map

![GFED Regions](../images/GFED-regions.png)

### Download Limits

- Maximum **1GB** per download request
- Use `split_large_dataset()` function for larger requests

## Utility Functions

The IRISCC utility functions are imported from the `fr.aeris.iriscc` module:
- `search_flights`: Search for flights matching given criteria
- `download_flights`: Download flight data as ZIP files (skips if already downloaded)
- `split_large_dataset`: Split large requests into smaller periods under 1GB
- `rank_flights_by_co`: Analyze and rank flights by CO concentration

## Search and Download

This section demonstrates how to:
1. **Define search parameters** once for all operations
2. **Search** to verify data availability and size
3. **Download L1, L2, and L4** data with the same parameters
4. **Analyze** CO data and rank flights by concentration
5. **Visualize** the most interesting flights

In [ ]:
# =============================================================================
# SEARCH PARAMETERS - Modify these according to your needs
# =============================================================================
# These parameters will be used for search, download, and visualization

SEARCH_PARAMS = {
    "start": "2025-07-23T00:00:00",
    "end": "2025-07-29T23:59:59",
    "bbox": "-180,-90,180,90",    # Global coverage (minLon,minLat,maxLon,maxLat)
    "region": None                 # Optional: ["BONA", "EURO"] to filter by region
}

print("Search parameters configured:")
print(json.dumps(SEARCH_PARAMS, indent=2))

In [ ]:
# Search to verify data availability
# This helps confirm the number of flights and total size before downloading

search_result = search_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L1",  # Search L1 first (preferred for CO data)
    region=SEARCH_PARAMS["region"]
)

if search_result:
    print("\nQuery details:")
    print(json.dumps(search_result['query'], indent=2))

### Download L1, L2, and L4 data

We download all three data levels with the same search parameters:
- **L2**: Validated data (preferred for CO)
- **L1**: Raw measurements (fallback when L2 has no valid CO). Note that these data aren't fully calibrated.
- **L4**: Regional CO contributions (for visualization)

In [ ]:
# Download L2 data (preferred for CO)

print("=" * 60)
print("Downloading L2 data...")
print("=" * 60)

zip_file_l2 = download_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L2",
    region=SEARCH_PARAMS["region"]
)

print(f"\nL2 ZIP file: {zip_file_l2}")

# Download L1 data (fallback when L2 has no valid CO)

print("\n" + "=" * 60)
print("Downloading L1 data...")
print("=" * 60)

zip_file_l1 = download_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L1",
    region=SEARCH_PARAMS["region"]
)

print(f"\nL1 ZIP file: {zip_file_l1}")

In [ ]:
# Download L4 data (regional CO contributions for visualization)

print("\n" + "=" * 60)
print("Downloading L4 data...")
print("=" * 60)

zip_file_l4 = download_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L4",
    region=SEARCH_PARAMS["region"],
    l4_type=["CO_CONTRIBUTIONS"]
)

print(f"\nL4 ZIP file: {zip_file_l4}")

### Analyze CO data across all flights

Now we use `rank_flights_by_co()` to:
1. Open each L2 file and check if **CO_P1** contains valid data
2. If L2 has no valid CO (variable missing, all NaN, or all -9999), fall back to the matching **L1 file**
3. Record the **max** and **mean** CO concentration for each flight
4. Sort all flights by **max CO descending** and display the **top 25**

This identifies the most interesting flights for further visualization.

In [ ]:
# Analyze all flights and rank by CO concentration
#
# This processes every L2 file in the ZIP, checks for valid CO_P1 data,
# and falls back to L1 when L2 has no CO data

print("=" * 60)
print("Analyzing CO data across all flights...")
print("=" * 60)

ranked_flights = rank_flights_by_co(zip_file_l2, zip_file_l1)

# Display the top 25 flights
top_n = 25
top_flights = ranked_flights[:top_n]

def fmt_avail(v):
    if v is None:
        return "-"
    return f"{v * 100:.0f}%"

print(f"\n{'=' * 210}")
print(f"  TOP {top_n} FLIGHTS BY MAX CO CONCENTRATION")
print(f"{'=' * 210}")
print(f"{'Rank':>4}  {'Flight ID':<20} {'Phase':<10} {'Airport':<25} {'Max':>8} {'Alt Max':>8} {'Mean':>8} {'Median':>8} {'P25':>8} {'P75':>8} {'Source':>6} {'CO Avail':>9}  {'Other Species':<30}")
print(f"{'':>4}  {'':20} {'':10} {'':25} {'(ppb)':>8} {'(m)':>8} {'(ppb)':>8} {'(ppb)':>8} {'(ppb)':>8} {'(ppb)':>8}")
print(f"{'-' * 210}")

for i, flight in enumerate(top_flights, 1):
    airport_short = flight['airport'][:25] if len(flight['airport']) > 25 else flight['airport']
    species = flight.get('other_species', '')
    co_avail_str = fmt_avail(flight.get('co_availability'))
    alt_max = flight.get('alt_max_co', 0.0)
    print(f"{i:>4}  {flight['flight_id']:<20} {flight['phase']:<10} {airport_short:<25} {flight['max_co']:>8.1f} {alt_max:>8.0f} {flight['mean_co']:>8.1f} {flight['median_co']:>8.1f} {flight['p25_co']:>8.1f} {flight['p75_co']:>8.1f} {flight['source']:>6} {co_avail_str:>9}  {species:<30}")

print(f"{'-' * 210}")
print("\nSource legend: L2 = validated data, L1 = raw data (L2 had no valid CO)")
print("CO Avail: percentage of valid CO_P1 data points (from NetCDF availability attribute)")
print(f"Total flights analyzed: {len(ranked_flights)}")

# Export all ranked flights to CSV
df = pd.DataFrame(ranked_flights)
csv_path = "../downloads/ranked_flights_by_co.csv"
df.to_csv(csv_path, index=False)
print(f"\nCSV exported: {csv_path}")

## Data Visualization

Now we visualize the CO profiles for the top flights. The plots show CO_P1 vs altitude.

In [ ]:
# =============================================================================
# PLOT SELECTION - Choose which flights to plot
# =============================================================================
# Option 1: Plot the top 6 flights (default)
#   PLOT_SELECTION = None
#
# Option 2: Plot specific flights by their rank in the table above
PLOT_SELECTION = [1, 3, 7, 12, 20, 25]
#PLOT_SELECTION = None  # Set to a list of ranks to pick specific flights

if PLOT_SELECTION is not None:
    plotted_flights = [ranked_flights[r - 1] for r in PLOT_SELECTION if r - 1 < len(ranked_flights)]
else:
    plotted_flights = ranked_flights[:6]

n_plots = len(plotted_flights)

# Plot CO profiles
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

print(f"Plotting CO profiles for {n_plots} flights...\n")

for i, flight in enumerate(plotted_flights):
    ax = axes[i]
    flight_id = flight['flight_id']
    airport = flight['airport']
    source = flight['source']
    rank = PLOT_SELECTION[i] if PLOT_SELECTION else i + 1

    try:
        if source == "L2" and flight.get('l2_filename'):
            with zipfile.ZipFile(zip_file_l2, 'r') as z:
                with z.open(flight['l2_filename']) as f:
                    ds = xr.open_dataset(io.BytesIO(f.read()))
        else:
            with zipfile.ZipFile(zip_file_l1, 'r') as z:
                with z.open(flight['l1_filename']) as f:
                    ds = xr.open_dataset(io.BytesIO(f.read()))

        if 'CO_P1' in ds and 'baro_alt_AC' in ds:
            altitude = ds['baro_alt_AC'].values.astype(float)
            ax.plot(ds['CO_P1'], altitude, color='grey', linewidth=2, alpha=0.8)

        ds.close()

    except Exception as e:
        print(f"  Error plotting flight {flight_id}: {e}")

    airport_str = f" ({airport})" if airport and airport != 'N/A' else ""
    ax.set_title(f"#{rank} - {flight_id}{airport_str}\nMax CO: {flight['max_co']:.1f} ppb [{source}]", fontsize=9)
    ax.set_xlabel('CO (ppb)', fontsize=8)
    ax.set_ylabel('Altitude (m)', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='both', labelsize=7)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top Flights by Maximum CO Concentration (CO_P1)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nCO profile plots complete!")
print("Red = L2 (validated), Pink = L1 (raw fallback)")

### Plot CO contributions by region for the top flights

For the flights with the highest CO concentration, we plot:
- The measured CO (CO_P1 from L2 or L1) as a reference line
- The L4 regional fire contributions (CO_GFAS_*) showing which GFED regions contributed most

This helps identify the geographic origin of the CO detected during the flight.

In [ ]:
# Plot CO contributions by region for the selected flights
# (uses plotted_flights from the cell above)
#
# X axis: CO (ppb), Y axis: Altitude (m)
# All 14 GFED regions are checked; only those with max > 0.1 ppb are plotted

n_plots = len(plotted_flights)

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()

all_regions = {
    'AUST': 'Australia',
    'BOAS': 'Boreal Asia',
    'BONA': 'Boreal N. America',
    'CEAM': 'Central America',
    'CEAS': 'Central Asia',
    'EQAS': 'Equatorial Asia',
    'EURO': 'Europe',
    'MIDE': 'Middle East',
    'NHAF': 'N. Hemisphere Africa',
    'NHSA': 'N. Hemisphere S. America',
    'SEAS': 'Southeast Asia',
    'SHAF': 'S. Hemisphere Africa',
    'SHSA': 'S. Hemisphere S. America',
    'TENA': 'Temperate N. America'
}

COLOR_HEX_BY_GFED4_REGION = {
    'BONA': '#3460ff',
    'TENA': '#ffab00',
    'CEAM': '#fe01fd',
    'NHSA': '#d8b6ff',
    'SHSA': '#956aff',
    'EURO': '#54ffff',
    'MIDE': '#488db1',
    'NHAF': '#547818',
    'SHAF': '#ff547e',
    'BOAS': '#ffff7e',
    'CEAS': '#aa8449',
    'SEAS': '#00d800',
    'EQAS': '#00acff',
    'AUST': '#e5ffbb',
}

print(f"Plotting regional CO contributions for {n_plots} flights...\n")

for i, flight in enumerate(plotted_flights):
    ax = axes[i]
    flight_id = flight['flight_id']
    phase = flight['phase']
    source = flight['source']
    airport = flight['airport']
    rank = PLOT_SELECTION[i] if PLOT_SELECTION else i + 1

    print(f"--- Flight #{rank}: {flight_id} ({phase}) | Source: {source} | Max CO: {flight['max_co']:.1f} ppb ---")

    ds_co = None
    ds_l4 = None

    try:
        # Load measured CO data (from L2 or L1) - in memory
        if source == "L2" and flight.get('l2_filename'):
            with zipfile.ZipFile(zip_file_l2, 'r') as z:
                with z.open(flight['l2_filename']) as f:
                    ds_co = xr.open_dataset(io.BytesIO(f.read()))
        else:
            with zipfile.ZipFile(zip_file_l1, 'r') as z:
                with z.open(flight['l1_filename']) as f:
                    ds_co = xr.open_dataset(io.BytesIO(f.read()))

        # Load matching L4 file - in memory
        with zipfile.ZipFile(zip_file_l4, 'r') as z:
            matching_l4 = [f for f in z.namelist() if flight_id in f and (phase == 'N/A' or f.endswith(f"-{phase}.nc4"))]
            if matching_l4:
                with z.open(matching_l4[0]) as f:
                    ds_l4 = xr.open_dataset(io.BytesIO(f.read()))

        # Get altitude
        altitude = None
        if ds_co is not None and 'baro_alt_AC' in ds_co:
            altitude = ds_co['baro_alt_AC'].values.astype(float)

        # Plot measured CO
        if ds_co is not None and 'CO_P1' in ds_co and altitude is not None:
            co_measured = ds_co['CO_P1'].values.astype(float)
            ax.plot(co_measured, altitude,
                    color='grey', linewidth=2,
                    label=f'{source} CO', alpha=0.8)

        # Plot regional fire contributions from L4
        if ds_l4 is not None and altitude is not None:
            alt_l4 = ds_l4['baro_alt_AC'].values.astype(float) if 'baro_alt_AC' in ds_l4 else altitude

            for region, name in all_regions.items():
                var_name = f'CO_GFAS_{region}'
                if var_name in ds_l4:
                    co_contrib = ds_l4[var_name].values.astype(float)
                    valid = co_contrib[~np.isnan(co_contrib)]
                    if len(valid) > 0 and np.max(valid) > 0.1:
                        region_color = COLOR_HEX_BY_GFED4_REGION.get(region, 'gray')
                        ax.plot(co_contrib, alt_l4, linewidth=1.5, label=region, alpha=0.8, color=region_color)

    except Exception as e:
        print(f"  Error: {e}")
    finally:
        if ds_co is not None:
            ds_co.close()
        if ds_l4 is not None:
            ds_l4.close()

    airport_short = airport[:25] if len(airport) > 25 else airport
    airport_str = f" ({airport_short})" if airport and airport != 'N/A' else ""
    phase_str = f" ({phase})" if phase and phase != 'N/A' else ""
    ax.set_title(f"#{rank} - {flight_id}{phase_str}{airport_str}\n"
                 f"Max CO: {flight['max_co']:.1f} ppb [{source}]", fontsize=9)
    ax.set_xlabel('CO (ppb)', fontsize=8)
    ax.set_ylabel('Altitude (m)', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=12)
    ax.tick_params(axis='both', labelsize=7)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top Flights - CO Fire Contributions by Region (GFAS)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nRegional contribution plots complete!")
print("Grey line: measured CO | Colored lines: L4 regional fire contributions (GFAS)")